In [ ]:
# the following is added due to a warning
import os
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
from keras.utils import image_dataset_from_directory as LdImg
from keras.layers import Rescaling, RandomFlip, Resizing,Conv2D,MaxPooling2D,Flatten,Dense,Input
from keras.models import Sequential
from keras.saving import save_model
from matplotlib import pyplot as plt
from sklearn.metrics import accuracy_score
import tensorflow as tf
import numpy as np

In [ ]:
#the TrainData is an tf.data.Dataset. It has two fields, images and labels(when label_mode ~=none)
#images has dimension [batch_size , image_size(0) , image_size(1) , channels] 
# value of channesl is denpedent from the color_mode, e.g. rgb menas 3 channels
# labels are generated from the directory structur
TrainData = LdImg("dataset/training_set", labels="inferred",
    label_mode="int",
    color_mode="rgb",
    shuffle = True,
    batch_size=32,
    image_size=(256, 256),
    crop_to_aspect_ratio=True)
print(TrainData.element_spec)

In [ ]:
#plot TrainData to learn the struct of TrainData
figure=plt.figure()
for images, labels in TrainData:
    i = int(1)
    for image, label in zip(images,labels):
        plt.subplot(4,6,i)
        plt.imshow(image.numpy().astype("uint8"))
        plt.title(label.numpy())
        plt.axis('off')
        if i<7:
            i += 1 
        else:
            break
    if i>=7:
        break
plt.show()      
        

In [ ]:
# creat a layer stacks to define the preprocessing
PreProcessing = Sequential([
    #rescales input values in [0 1]
    Rescaling(1./255), 
    #resizing the image to 50x50
    Resizing(50,50)
])
#summarize the preprocessing layer
PreProcessing.summary()

In [ ]:
# use map-fun to preprocess the image data
#the traindata has 2 fields, the preporcessing is only appled on images, labels are not changed
TrainDataPreProcd = TrainData.map(lambda images, labels: (PreProcessing(images),labels))

In [ ]:
figure=plt.figure()
for images, labels in TrainDataPreProcd:
    i = int(1)
    for image,label in zip(images,labels):
        plt.subplot(4,6,i)
        plt.imshow((image.numpy()*255).astype("uint8"))
        plt.title(label.numpy())
        plt.axis('off')
        if i<7:
            i += 1 
        else:
            break
    if i>=7:
        break
plt.show() 

In [ ]:
#create the cnn model
CNNModel = Sequential()
# Define the input shape instead specify it direction in the first convolution layer
CNNModel.add(Input(shape=(50, 50, 3))) 
#add convolution layer
CNNModel.add(Conv2D(32,(3,3),activation='relu'))
#add pooling layer
CNNModel.add(MaxPooling2D(pool_size=(2,2)))
#add another convolution layer
CNNModel.add(Conv2D(32,(3,3),activation='relu'))
CNNModel.add(MaxPooling2D(pool_size=(2,2)))
#add flatten layer
CNNModel.add(Flatten())
#add fullly connected layer
CNNModel.add(Dense(units=128,activation='relu'))
CNNModel.add(Dense(units=1,activation='sigmoid'))

In [ ]:
#configure the cnn model
CNNModel.compile(optimizer='Adam',loss='binary_crossentropy',metrics=['accuracy'])

In [ ]:
CNNModel.summary()

In [ ]:
#train the model
CNNModel.fit(TrainDataPreProcd,epochs=25)

In [ ]:
# Batching train data is essential when training a TensorFlow Convolutional Neural Network (CNN) model.
# Batching helps in several ways:
# Efficiency: It allows the model to process multiple samples simultaneously, making better use of computational resources.
# Memory Management: It helps manage memory usage, especially when dealing with large datasets.
# Stability: It can improve the stability of the training process by averaging gradients over multiple samples.Batching helps in Stability, Memory Management and Efficiency:

# Why unbatch the train data
# The shuffle is set true, the load data are sorted randomly. 
# Ohterwise first train image with cats then dogs. This might be not good.
# After train I want to use the same data to check training results, prediction with train data
# Due to the train data is batched. It might be easy to handel the label when the data are unbatched
UnbatchedSet = TrainDataPreProcd.unbatch()

In [ ]:
PredResults = []
ExpectedResults = []
for image, label in UnbatchedSet:
    #predict the image
    #the model expect shape [bath_size, image_width, image_height, channels]
    #unbatched data doesn't have the 1st(bath_size) axis
    #therefore an additional axis is added tf.expand_dims
    pred = CNNModel.predict(tf.expand_dims(image, axis=0))
    #predicated results is a float between[0 1], ref to the activation function sigmoid
    #cast pred to int >0.5 becomes to 1, otherwise 0
    pred = (pred>0.5).astype(int)
    #the predict results is array due to the last dense layer
    #print(f'predicted {pred[0]} should be {label.numpy()}')
    PredResults.extend(pred[0])
    #add the label to the list on same place as prediction
    ExpectedResults.extend(np.array([label.numpy()]))

In [ ]:
ComparResults = list(zip(PredResults, ExpectedResults))

In [ ]:
Results = np.array(ComparResults).reshape(-1,2)
#print(Results)

In [ ]:
AccuracyScore = accuracy_score(Results[:,1],Results[:,0])
print(AccuracyScore)

In [ ]:
CNNModel.save("CNNModel.keras")